# 06 - Lines and Distances

## Goal

In this lesson we use the WDO spatial helpers (or a fallback distance function) to calculate many distances and draw lines from MSU to cities around the world.

We will:

- load a world-cities file
- filter the data down to a manageable number of cities
- calculate distances from MSU
- draw lines on a Folium map
- optionally write a GeoJSON output file

This lesson helps with **📏 Milestone 2**.


## Notes

The original starter code was checking for `countries.geojson`, but this lesson is really about the **world cities** files.

So this notebook fixes that and looks for:

- `world_cities_large.json`
- `world_cities_fixed.json`
- `world_cities_by_time-zone.json`

in the shared `data/` folder.


In [2]:
from pathlib import Path
import json
import random
import math
import sys

import folium
import pandas as pd

/Users/ricardoayala/Desktop/ricardoayala2510-Spatial-Data-Mapping/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Find the data folder and choose a city file

In [3]:
debug = False

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "data").exists()), None)

assert repo_root is not None, "❌ Could not find a parent folder containing a data directory."

data_dir = repo_root / "data"

if debug:
    print("Current Working Directory:", cwd)
    print("Repo Root:", repo_root)
    print("Data Directory:", data_dir)

candidates = [
    data_dir / "world_cities_large.json",
    data_dir / "world_cities_fixed.json",
    data_dir / "world_cities_by_time-zone.json",
]

target = next((p for p in candidates if p.exists()), None)

assert target is not None, (
    "❌ Could not find any expected world-cities file in the data folder.\n"
    f"Tried: {[str(p) for p in candidates]}"
)

print("Using file:")
print(target)

Using file:
/Users/ricardoayala/Desktop/ricardoayala2510-Spatial-Data-Mapping/assigments completed/02-Missile_Geometry_101/data/world_cities_fixed.json


## 2. Load WDO helpers if available, otherwise use a fallback

In [4]:
GEO_OK = False

try:
    from wdo.geo import haversine_km
    GEO_OK = True
    print("[INFO] Loaded haversine_km from wdo.geo")
except Exception as e:
    print(f"[WARN] Could not import wdo.geo.haversine_km: {e}")
    print("[WARN] Using local fallback function instead.")

    EARTH_RADIUS_KM = 6371.0088

    def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
        phi1 = math.radians(lat1)
        phi2 = math.radians(lat2)
        dphi = math.radians(lat2 - lat1)
        dlmb = math.radians(lon2 - lon1)

        a = (
            math.sin(dphi / 2) ** 2
            + math.cos(phi1) * math.cos(phi2) * math.sin(dlmb / 2) ** 2
        )
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        return EARTH_RADIUS_KM * c

[WARN] Could not import wdo.geo.haversine_km: No module named 'wdo'
[WARN] Using local fallback function instead.


## 3. Load the city data

In [5]:
with open(target, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Normalize the input into one flat list of city dictionaries
if isinstance(raw, list):
    cities = raw
elif isinstance(raw, dict):
    # world_cities_by_time-zone.json structure
    flat = []
    for tz, city_list in raw.items():
        for city in city_list:
            item = city.copy()
            item["time-zone"] = tz
            flat.append(item)
    cities = flat
else:
    raise ValueError("❌ Unexpected JSON structure for world cities file.")

print("Number of raw city records:", len(cities))
cities[:3]

Number of raw city records: 149102


[{'city-name': 'El Tarter',
  'lat': '42.57952',
  'lon': '1.65362',
  'time-zone': 'Europe/Andorra'},
 {'city-name': 'Sant Julia de Loria',
  'lat': '42.46372',
  'lon': '1.49129',
  'time-zone': 'Europe/Andorra'},
 {'city-name': 'Pas de la Casa',
  'lat': '42.54277',
  'lon': '1.73361',
  'time-zone': 'Europe/Andorra'}]

## 4. Clean and normalize fields

In [6]:
cleaned = []

for city in cities:
    try:
        name = city.get("city-name", city.get("name", "Unknown"))
        lat = float(city["lat"])
        lon = float(city["lon"])
        tz = city.get("time-zone", "Unknown")
        cleaned.append({
            "city-name": name,
            "lat": lat,
            "lon": lon,
            "time-zone": tz,
        })
    except Exception:
        # Skip broken records
        continue

print("Number of cleaned city records:", len(cleaned))
cleaned[:5]

Number of cleaned city records: 149102


[{'city-name': 'El Tarter',
  'lat': 42.57952,
  'lon': 1.65362,
  'time-zone': 'Europe/Andorra'},
 {'city-name': 'Sant Julia de Loria',
  'lat': 42.46372,
  'lon': 1.49129,
  'time-zone': 'Europe/Andorra'},
 {'city-name': 'Pas de la Casa',
  'lat': 42.54277,
  'lon': 1.73361,
  'time-zone': 'Europe/Andorra'},
 {'city-name': 'Ordino',
  'lat': 42.55623,
  'lon': 1.53319,
  'time-zone': 'Europe/Andorra'},
 {'city-name': 'les Escaldes',
  'lat': 42.50729,
  'lon': 1.53414,
  'time-zone': 'Europe/Andorra'}]

## 5. MSU reference point

These coordinates are near Midwestern State University / Bolin Hall in Wichita Falls.


In [7]:
MSU_NAME = "Midwestern State University"
MSU_LAT = 33.87372
MSU_LON = -98.51947

print(MSU_NAME, MSU_LAT, MSU_LON)

Midwestern State University 33.87372 -98.51947


## 6. Calculate distances to all cities

We compute the distance from MSU to every city in the cleaned dataset.


In [8]:
rows = []

for city in cleaned:
    d = haversine_km(MSU_LAT, MSU_LON, city["lat"], city["lon"])
    row = city.copy()
    row["distance_km"] = d
    rows.append(row)

df = pd.DataFrame(rows)
df.head()

,city-name,lat,lon,time-zone,distance_km
0,El Tarter,42.57952,1.65362,Europe/Andorra,8271.456386
1,Sant Julia de Loria,42.46372,1.49129,Europe/Andorra,8266.971551
2,Pas de la Casa,42.54277,1.73361,Europe/Andorra,8279.177671
3,Ordino,42.55623,1.53319,Europe/Andorra,8264.454270
4,les Escaldes,42.50729,1.53414,Europe/Andorra,8267.395396


In [9]:
print("Total cities with computed distances:", len(df))
df[["city-name", "time-zone", "distance_km"]].sort_values("distance_km").head(15)

Total cities with computed distances: 149102


,city-name,time-zone,distance_km
136616,Wichita Falls,America/Chicago,5.056442
136148,Iowa Park,America/Chicago,16.258031
136121,Holliday,America/Chicago,17.431576
135886,Burkburnett,America/Chicago,25.366452
136106,Henrietta,America/Chicago,30.585811
135806,Archer City,America/Chicago,32.439353
136002,Electra,America/Chicago,40.704982
135102,Walters,America/Chicago,57.455013
135107,Waurika,America/Chicago,58.117311
136352,Olney,America/Chicago,59.966267


## 7. Filter to a manageable set of cities

Drawing lines to *every* city is too much.

Here we keep cities that are:
- within a threshold distance from MSU
- then take the closest N cities

You can change either value.


In [10]:
DISTANCE_THRESHOLD_KM = 2500
MAX_CITIES = 30

subset = (
    df[df["distance_km"] <= DISTANCE_THRESHOLD_KM]
    .sort_values("distance_km")
    .head(MAX_CITIES)
    .copy()
)

print("Cities selected:", len(subset))
subset[["city-name", "time-zone", "distance_km"]].head(20)

Cities selected: 30


,city-name,time-zone,distance_km
136616,Wichita Falls,America/Chicago,5.056442
136148,Iowa Park,America/Chicago,16.258031
136121,Holliday,America/Chicago,17.431576
135886,Burkburnett,America/Chicago,25.366452
136106,Henrietta,America/Chicago,30.585811
135806,Archer City,America/Chicago,32.439353
136002,Electra,America/Chicago,40.704982
135102,Walters,America/Chicago,57.455013
135107,Waurika,America/Chicago,58.117311
136352,Olney,America/Chicago,59.966267


## 8. Bonus option: random sampling under a distance threshold

This is optional. It randomly chooses cities until the limit is reached.


In [11]:
RANDOM_THRESHOLD_KM = 10000
RANDOM_SAMPLE_SIZE = 20

random_pool = df[df["distance_km"] <= RANDOM_THRESHOLD_KM].copy()
random_subset = random_pool.sample(
    n=min(RANDOM_SAMPLE_SIZE, len(random_pool)),
    random_state=42
).sort_values("distance_km")

print("Random subset size:", len(random_subset))
random_subset[["city-name", "time-zone", "distance_km"]].head(10)

Random subset size: 20


,city-name,time-zone,distance_km
136275,Mart,America/Chicago,303.436671
133784,Wellsville,America/Chicago,848.148618
133510,Crystal City,America/Chicago,876.548713
145804,Sunset,America/Denver,1437.011637
95825,Cantera de Villagran,America/Mexico_City,1562.336165
95508,La Finca,America/Mexico_City,1669.763360
139304,Sandusky,America/Detroit,1722.309139
133156,Charlestown,America/New_York,2032.474817
144056,Modesto,America/Los_Angeles,2065.722536
59214,Meltham,Europe/London,7445.889219


## 9. Draw lines on a Folium map

This map draws:
- a marker at MSU
- one marker for each selected city
- one line from MSU to each selected city


In [12]:
m = folium.Map(location=[MSU_LAT, MSU_LON], zoom_start=4)

# Marker for MSU
folium.Marker(
    location=[MSU_LAT, MSU_LON],
    tooltip=MSU_NAME,
    popup=MSU_NAME,
    icon=folium.Icon(color="green", icon="info-sign")
).add_to(m)

# Add selected cities and lines
for _, row in subset.iterrows():
    city_lat = row["lat"]
    city_lon = row["lon"]
    city_name = row["city-name"]
    dist = row["distance_km"]

    folium.Marker(
        location=[city_lat, city_lon],
        tooltip=f"{city_name} ({dist:.1f} km)",
        popup=f"{city_name}<br>{dist:.1f} km",
        icon=folium.Icon(color="red", icon="info-sign")
    ).add_to(m)

    folium.PolyLine(
        locations=[[MSU_LAT, MSU_LON], [city_lat, city_lon]],
        weight=2,
        opacity=0.7,
        popup=f"{MSU_NAME} → {city_name}: {dist:.1f} km"
    ).add_to(m)

m

## 10. Write a GeoJSON file with the lines

This creates a GeoJSON **FeatureCollection** of line features from MSU to each selected city.


In [13]:
line_features = []

for _, row in subset.iterrows():
    line_features.append(
        {
            "type": "Feature",
            "properties": {
                "from": MSU_NAME,
                "to": row["city-name"],
                "distance_km": round(float(row["distance_km"]), 3),
                "time-zone": row["time-zone"],
                "stroke": "#ff00ff",
            },
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [MSU_LON, MSU_LAT],
                    [float(row["lon"]), float(row["lat"])],
                ],
            },
        }
    )

geojson_lines = {
    "type": "FeatureCollection",
    "features": line_features,
}

out_path = data_dir / "msu_to_selected_cities.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(geojson_lines, f, indent=2)

print("Wrote GeoJSON lines file:")
print(out_path)

Wrote GeoJSON lines file:
/Users/ricardoayala/Desktop/ricardoayala2510-Spatial-Data-Mapping/assigments completed/02-Missile_Geometry_101/data/msu_to_selected_cities.geojson


## 11. Summary

This notebook:

- loaded a world-cities dataset
- computed distances from MSU
- filtered to a manageable subset
- drew lines on a Folium map
- wrote a GeoJSON line file

You can experiment by changing:
- the distance threshold
- the max number of cities
- the filtering rule
- whether to use the random subset instead
